In [ ]:
from agents import Agent, Runner, function_tool, ItemHelpers


@function_tool
def get_weather(city: str):
    """도시별 날씨 받아오기"""
    return f"30 degrees"

agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions.",
    tools=[get_weather],
)

steam = Runner.run_streamed(agent, "안녕 일본의 수도 날씨는 어때??")

async for event in steam.stream_events():    
    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print("Agent updated to", event.new_agent.name)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict())
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item))


    print("="*20)

Agent updated to Assistant Agent
{'arguments': '{"city":"도쿄"}', 'call_id': 'call_b2Zwh7KFSuj59WJfXpEEb7UM', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_08d0c6d472882a6d006985a8f143bc8190af0b15eb3105e677', 'status': 'completed'}
30 degrees
일본의 수도 도쿄의 현재 날씨는 30도입니다. 덥네요! 다른 정보가 궁금하신가요?


In [ ]:
from agents import Agent, Runner, function_tool, ItemHelpers

@function_tool
def get_weather(city: str):
    """도시별 날씨 받아오기"""
    return f"30 degrees"

agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions.",
    tools=[get_weather],
)

steam = Runner.run_streamed(agent, "안녕 일본의 수도 날씨는 어때??")

message = ""
args = ""

async for event in steam.stream_events():    
    if event.type == "raw_response_event":
        event_type = event.data.type
        if event_type == "response.output_text.delta":
            message += event.data.delta
            print(message)
        elif event_type == "response.function_call_arguments.delta":
            args += event.data.delta
            print(args)
        elif event_type == "response.completed":
            message = ""
            args = ""



In [3]:

from agents import Agent, Runner, function_tool, SQLiteSession 

session = SQLiteSession("user_2", "ai-memory.db")

@function_tool
def get_weather(city: str):
    """도시별 날씨 받아오기"""
    return f"30 degrees"

agent = Agent(
    name="Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions.",
    tools=[get_weather],
)

In [19]:
result = await Runner.run(
    agent, 
    "근데 내 이름이 뭐였는지 기억하고 있어?",
    session=session
    )

print(result.final_output)

네가 이전에 이름을 말해 준 적이 있다면 기억할 수 있지만, 이번 대화에서 아직 네 이름을 알려주지 않았어. 네 이름을 알려주면 앞으로 기억해서 부를 수 있어! 이름이 뭐야?


In [18]:
await session.pop_item()

{'role': 'user', 'content': 'My name is sam'}

In [21]:
from agents import Agent, function_tool, SQLiteSession 

session = SQLiteSession("user_2", "ai-memory.db")

@function_tool
def get_weather(city: str):
    """도시별 날씨 받아오기"""
    return f"30 degrees"

geaography_agent = Agent(
    name="Geo Expert Agent",
    instructions="너는 지리학 전문가야 지리와 관련된 질문에 답변을 해줘" ,
    handoff_description="지리학 관련 질문에 답할 때 이걸 사용해"
)

economics_agent = Agent(
    name="Economics Expert Agent",
    instructions="너는 경제학 전문가야 경제와 관련된 질문에 답변을 해줘" ,
    handoff_description="경제학 관련 질문에 답할 때 이걸 사용해"
)

main_agent = Agent(
    name="Main Agent",
    instructions="너는 사용자 인터페이스 angent고, 사용자의 질문에 답변하는 데에 있어서 가장 적합한 agent에게 작업을 넘겨줘",
    handoffs=[
        economics_agent,
        geaography_agent
    ]
)

In [ ]:
result = await Runner.run(
    main_agent, 
    "한국에서 경제 상황 너가 생각할 때 어떤거 같아??",
    session=session
    )

print(result.last_agent.name)
print(result.final_output)

Geo Expert Agent
한국에서 두 번째로 큰 도시는 "부산광역시"입니다.

- 인구 기준: 서울특별시가 가장 크고, 그 다음이 부산광역시입니다.
- 면적 기준: 울산광역시, 경기도 등 더 넓은 광역 지자체도 있지만, "도시"로서 인구와 규모를 모두 고려할 때 부산이 일반적으로 두 번째로 큰 도시로 분류됩니다.

부산은 해운대, 광안리 해수욕장, 자갈치 시장 등으로 유명한 항구도시입니다.
